# Stock Market Prediction with Deep Learning
### LSTM · GRU · Bidirectional LSTM + Attention · Monte Carlo Uncertainty

This notebook builds a complete pipeline for forecasting stock closing prices:

- **Live market data** fetched directly from Yahoo Finance for any ticker
- **20 technical indicator features** — RSI, MACD, Bollinger Bands, ATR, OBV, moving averages and more
- **Three deep learning architectures** trained and compared side by side
- **Monte Carlo Dropout** for honest uncertainty estimation (90% confidence intervals)
- **Five evaluation metrics** — RMSE, MAE, MAPE, R², Directional Accuracy
- **Interactive Plotly charts** for predictions, residuals, and model comparison


## 1 · Install Dependencies

In [13]:
# Only installs what Colab doesn't already have
!pip install yfinance plotly -q


## 2 · Imports

In [14]:
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print('✅ All imports successful')


TensorFlow : 2.20.0
NumPy      : 2.0.2
✅ All imports successful


## 3 · Configuration
Change `TICKER` to any Yahoo Finance symbol — AAPL, GOOGL, TSLA, etc.

In [15]:
TICKER       = 'MSFT'        # ← change to any ticker
START_DATE   = '2015-01-01'
END_DATE     = None          # None = today
SEQ_LEN      = 60            # look-back window in trading days
EPOCHS       = 100
BATCH_SIZE   = 32
MC_ITER      = 50            # Monte Carlo dropout passes
TRAIN_RATIO  = 0.80


## 4 · Technical Indicator Engineering

In [16]:
def add_technical_indicators(df):
    close = df['Close']

    # Moving averages
    df['SMA_10']      = close.rolling(10).mean()
    df['SMA_30']      = close.rolling(30).mean()
    df['EMA_12']      = close.ewm(span=12, adjust=False).mean()
    df['EMA_26']      = close.ewm(span=26, adjust=False).mean()

    # MACD
    df['MACD']        = df['EMA_12'] - df['EMA_26']
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_hist']   = df['MACD'] - df['MACD_signal']

    # Bollinger Bands
    bb_mid            = close.rolling(20).mean()
    bb_std            = close.rolling(20).std()
    df['BB_upper']    = bb_mid + 2 * bb_std
    df['BB_lower']    = bb_mid - 2 * bb_std
    df['BB_width']    = (df['BB_upper'] - df['BB_lower']) / bb_mid

    # RSI (14-day)
    delta             = close.diff()
    gain              = delta.clip(lower=0).rolling(14).mean()
    loss              = (-delta.clip(upper=0)).rolling(14).mean()
    df['RSI']         = 100 - (100 / (1 + gain / (loss + 1e-10)))

    # ATR
    hl                = df['High'] - df['Low']
    hpc               = (df['High'] - close.shift(1)).abs()
    lpc               = (df['Low']  - close.shift(1)).abs()
    df['ATR']         = pd.concat([hl, hpc, lpc], axis=1).max(axis=1).rolling(14).mean()

    # On-Balance Volume
    df['OBV']         = (np.sign(close.diff()) * df['Volume']).fillna(0).cumsum()

    # Return & volatility
    df['Return']      = close.pct_change()
    df['Volatility']  = df['Return'].rolling(10).std()

    return df

print('✅ Technical indicator functions ready')


✅ Technical indicator functions ready


## 5 · Fetch & Prepare Data

In [17]:
def prepare_data(ticker, start, end, seq_len, train_ratio):
    print(f'Downloading {ticker} data...')
    raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)

    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(1)
    # Remove any duplicate columns yfinance may produce
    raw = raw.loc[:, ~raw.columns.duplicated()]
    raw.dropna(inplace=True)

    df = add_technical_indicators(raw.copy())
    df.dropna(inplace=True)

    feature_cols = [
        'Open', 'High', 'Low', 'Close', 'Volume',
        'SMA_10', 'SMA_30', 'EMA_12', 'EMA_26',
        'MACD', 'MACD_signal', 'MACD_hist',
        'BB_upper', 'BB_lower', 'BB_width',
        'RSI', 'ATR', 'OBV', 'Return', 'Volatility',
    ]

    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X_scaled = scaler_X.fit_transform(df[feature_cols])
    y_scaled = scaler_y.fit_transform(df[['Close']])

    X_seq, y_seq, dates = [], [], []
    for i in range(seq_len, len(df)):
        X_seq.append(X_scaled[i - seq_len: i])
        y_seq.append(y_scaled[i, 0])
        dates.append(df.index[i])

    X_seq   = np.array(X_seq,  dtype=np.float32)
    y_seq   = np.array(y_seq,  dtype=np.float32)
    dates   = np.array(dates)

    split        = int(len(X_seq) * train_ratio)
    return {
        'X_train': X_seq[:split],  'X_test': X_seq[split:],
        'y_train': y_seq[:split],  'y_test': y_seq[split:],
        'dates_train': dates[:split], 'dates_test': dates[split:],
        'scaler_y': scaler_y, 'raw_df': df, 'feature_cols': feature_cols,
    }

data         = prepare_data(TICKER, START_DATE, END_DATE, SEQ_LEN, TRAIN_RATIO)
X_train      = data['X_train']
X_test       = data['X_test']
y_train      = data['y_train']
y_test       = data['y_test']
scaler_y     = data['scaler_y']
dates_test   = data['dates_test']
input_shape  = (X_train.shape[1], X_train.shape[2])
y_test_price = scaler_y.inverse_transform(y_test.reshape(-1, 1)).squeeze()

print(f'Ticker       : {TICKER}')
print(f'Input shape  : {input_shape}')
print(f'Train samples: {len(X_train)}')
print(f'Test  samples: {len(X_test)}')
print(f'Date range   : {data["dates_train"][0].strftime("%Y-%m-%d")} → {data["dates_test"][-1].strftime("%Y-%m-%d")}')


Ticker       : MSFT
Input shape  : (60, 20)
Train samples: 2236
Test  samples: 560
Date range   : 2015-05-12 → 2026-06-24


## 6 · Feature Correlation with Close Price

In [18]:
# Use only the feature columns we defined — avoids any duplicate Close issue
feat_df = data['raw_df'][data['feature_cols']].copy()
close_series = data['raw_df']['Close'].copy()
if isinstance(close_series, pd.DataFrame):
    close_series = close_series.iloc[:, 0]
feat_df['_Close'] = close_series.values

corr = feat_df.corr()[['_Close']].drop('_Close')
corr.columns = ['Close']
corr = corr.sort_values('Close', ascending=True)

fig = go.Figure(go.Bar(
    x=corr['Close'], y=corr.index, orientation='h',
    marker=dict(color=corr['Close'], colorscale='RdBu', cmid=0),
))
fig.update_layout(
    title=f'{TICKER} — Feature Correlation with Close Price',
    xaxis_title='Pearson r', template='plotly_white', height=550,
)
fig.show()


## 7 · Model Definitions
Three architectures: **Vanilla LSTM**, **GRU**, and **Bidirectional LSTM + Attention**.

In [19]:
# ── Attention Layer ──────────────────────────────────────────────────────
class BahdanauAttention(layers.Layer):
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, hidden_states):
        score   = self.V(tf.nn.tanh(self.W(hidden_states)))
        weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(weights * hidden_states, axis=1)
        return context, tf.squeeze(weights, -1)


# ── Vanilla LSTM ─────────────────────────────────────────────────────────
def build_vanilla_lstm(input_shape, units=64, dropout=0.2):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(units, return_sequences=True),
        layers.Dropout(dropout),
        layers.BatchNormalization(),
        layers.LSTM(units // 2),
        layers.Dropout(dropout),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),
    ], name='VanillaLSTM')
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='huber', metrics=['mae'])
    return model


# ── GRU ──────────────────────────────────────────────────────────────────
def build_gru(input_shape, units=64, dropout=0.2):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.GRU(units, return_sequences=True),
        layers.Dropout(dropout),
        layers.BatchNormalization(),
        layers.GRU(units // 2),
        layers.Dropout(dropout),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),
    ], name='GRU')
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='huber', metrics=['mae'])
    return model


# ── BiLSTM + Attention ────────────────────────────────────────────────────
def build_bilstm_attention(input_shape, units=64, dropout=0.2):
    inp = layers.Input(shape=input_shape)
    x   = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(inp)
    x   = layers.Dropout(dropout)(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Bidirectional(layers.LSTM(units // 2, return_sequences=True))(x)
    x   = layers.Dropout(dropout)(x)
    ctx, _ = BahdanauAttention(64)(x)
    x   = layers.Dense(64, activation='relu')(ctx)
    x   = layers.Dropout(dropout / 2)(x)
    x   = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1)(x)
    model = Model(inputs=inp, outputs=out, name='BiLSTM_Attention')
    model.compile(optimizer=keras.optimizers.Adam(5e-4), loss='huber', metrics=['mae'])
    return model


# ── Shared callbacks ──────────────────────────────────────────────────────
def get_callbacks():
    return [
        callbacks.EarlyStopping(monitor='val_loss', patience=15,
                                restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                   patience=7, min_lr=1e-6, verbose=1),
    ]


# ── Monte Carlo Dropout ───────────────────────────────────────────────────
def mc_predict(model, X, n=50):
    preds = np.stack(
        [model(X, training=True).numpy().squeeze() for _ in range(n)], axis=0
    )
    return preds.mean(0), np.percentile(preds, 5, axis=0), np.percentile(preds, 95, axis=0)


print('✅ Model definitions ready')


✅ Model definitions ready


## 8 · Train All Three Models


In [31]:
COLORS = {
    'VanillaLSTM':      '#4C72B0',
    'GRU':              '#DD8452',
    'BiLSTM_Attention': '#55A868',
}

model_builders = {
    'VanillaLSTM':      lambda: build_vanilla_lstm(input_shape),
    'GRU':              lambda: build_gru(input_shape),
    'BiLSTM_Attention': lambda: build_bilstm_attention(input_shape),
}

trained    = {}
histories  = {}
preds_price = {}
mc_bands   = {}
metrics    = []

for name, builder in model_builders.items():
    print(f'\n{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    model = builder()
    hist  = model.fit(
        X_train, y_train,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_split=0.1,
        callbacks=get_callbacks(),
        verbose=1, shuffle=False,
    )
    trained[name]   = model
    histories[name] = hist.history

    # Point predictions → price
    y_pred_s = model.predict(X_test, verbose=0).squeeze()
    y_pred_p = scaler_y.inverse_transform(y_pred_s.reshape(-1, 1)).squeeze()
    preds_price[name] = y_pred_p

    # MC uncertainty bands → price
    mc_m, mc_lo, mc_hi = mc_predict(model, X_test, MC_ITER)
    mc_bands[name] = (
        scaler_y.inverse_transform(mc_m.reshape(-1,1)).squeeze(),
        scaler_y.inverse_transform(mc_lo.reshape(-1,1)).squeeze(),
        scaler_y.inverse_transform(mc_hi.reshape(-1,1)).squeeze(),
    )

    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test_price, y_pred_p))
    mae  = mean_absolute_error(y_test_price, y_pred_p)
    mape = np.mean(np.abs((y_test_price - y_pred_p) / (np.abs(y_test_price) + 1e-8))) * 100
    r2   = r2_score(y_test_price, y_pred_p)
    da   = (np.sign(np.diff(y_test_price)) == np.sign(np.diff(y_pred_p))).mean() * 100
    metrics.append({'Model': name, 'RMSE': rmse, 'MAE': mae,
                    'MAPE (%)': mape, 'R²': r2, 'Dir. Acc. (%)': da})
    print(f'  RMSE={rmse:.3f}  MAE={mae:.3f}  MAPE={mape:.2f}%  R²={r2:.4f}  DirAcc={da:.1f}%')

print('\n✅ All models trained')



  VanillaLSTM
Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - loss: 0.0124 - mae: 0.1212 - val_loss: 0.0193 - val_mae: 0.1872 - learning_rate: 0.0010
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0104 - mae: 0.1098 - val_loss: 0.0635 - val_mae: 0.3487 - learning_rate: 0.0010
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0110 - mae: 0.1208 - val_loss: 0.0334 - val_mae: 0.2466 - learning_rate: 0.0010
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0126 - mae: 0.1256 - val_loss: 0.0064 - val_mae: 0.0971 - learning_rate: 0.0010
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0153 - mae: 0.1408 - val_loss: 0.0035 - val_mae: 0.0729 - learning_rate: 0.0010
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0237 - mae: 0.1794 - val_loss: 0.0471 - val_mae: 0.2987 - learning_rate: 0.0010
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0146 - mae: 0.1428 - val_loss: 0.0239 - val_mae: 0.1951 - learning_rate

## 9 · Results Table

In [30]:
results_df = pd.DataFrame(metrics).set_index('Model').round(4)
print(results_df.to_string())
results_df


                      RMSE       MAE   MAPE (%)       R²  Dir. Acc. (%)
Model                                                                  
VanillaLSTM       153.8034  147.1085  33.059601 -11.3450        49.5528
GRU                25.6769   20.4226   4.588700   0.6559        52.9517
BiLSTM_Attention   83.9172   66.8090  14.389000  -2.6750        51.6995


,RMSE,MAE,MAPE (%),R²,Dir. Acc. (%)
Model,,,,,
VanillaLSTM,153.8034,147.1085,33.059601,-11.3450,49.5528
GRU,25.6769,20.4226,4.588700,0.6559,52.9517
BiLSTM_Attention,83.9172,66.8090,14.389000,-2.6750,51.6995


## 10 · Training & Validation Loss

In [29]:
fig = go.Figure()
for name, hist in histories.items():
    ep = list(range(1, len(hist['loss']) + 1))
    c  = COLORS[name]
    fig.add_trace(go.Scatter(x=ep, y=hist['loss'],     mode='lines',
                             name=f'{name} train', line=dict(color=c)))
    fig.add_trace(go.Scatter(x=ep, y=hist['val_loss'], mode='lines',
                             name=f'{name} val',   line=dict(color=c, dash='dash')))
fig.update_layout(title='Training & Validation Loss (Huber)',
                  xaxis_title='Epoch', yaxis_title='Loss',
                  template='plotly_white', height=450)
fig.show()


## 11 · Predicted vs Actual Close Price
Shaded bands show the 90% Monte Carlo confidence interval for each model.

In [28]:
fig = go.Figure()

def hex_to_rgba(hex_color, alpha=0.12):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

fig.add_trace(go.Scatter(x=dates_test, y=y_test_price, mode='lines',
                         name='Actual', line=dict(color='#C44E52', width=2)))

for name, y_pred in preds_price.items():
    c = COLORS[name]
    _, lo, hi = mc_bands[name]
    fig.add_trace(go.Scatter(
        x=list(dates_test) + list(dates_test[::-1]),
        y=list(hi) + list(lo[::-1]),
        fill='toself', fillcolor=hex_to_rgba(c, 0.12),
        line=dict(color='rgba(0,0,0,0)'),
        name=f'{name} 90% CI', showlegend=True,
    ))
    fig.add_trace(go.Scatter(x=dates_test, y=y_pred, mode='lines',
                             name=name, line=dict(color=c, width=1.5)))

fig.update_layout(
    title=f'{TICKER} — Predicted vs Actual Closing Price (Test Set)',
    xaxis_title='Date', yaxis_title='Price (USD)',
    template='plotly_white', hovermode='x unified', height=520,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()


## 12 · Residuals (Actual − Predicted)

In [27]:
fig = go.Figure()
for name, y_pred in preds_price.items():
    fig.add_trace(go.Scatter(x=dates_test, y=y_test_price - y_pred,
                             mode='lines', name=name,
                             line=dict(color=COLORS[name])))
fig.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.4)
fig.update_layout(title='Prediction Residuals',
                  xaxis_title='Date', yaxis_title='Residual (USD)',
                  template='plotly_white', height=380)
fig.show()


## 13 · Model Comparison Radar Chart

In [26]:
metric_cols = ['RMSE', 'MAE', 'MAPE (%)', 'R²', 'Dir. Acc. (%)']
norm = results_df[metric_cols].copy()

# Normalise — invert error metrics so higher = better everywhere
for col in ['RMSE', 'MAE', 'MAPE (%)']:
    mn, mx = norm[col].min(), norm[col].max()
    norm[col] = 1 - (norm[col] - mn) / (mx - mn + 1e-10)
for col in ['R²', 'Dir. Acc. (%)']:
    mn, mx = norm[col].min(), norm[col].max()
    norm[col] = (norm[col] - mn) / (mx - mn + 1e-10)

fig = go.Figure()
for name in norm.index:
    vals = norm.loc[name].tolist()
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=metric_cols + [metric_cols[0]],
        fill='toself', name=name, line=dict(color=COLORS[name]),
    ))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Model Comparison — higher is better on all axes',
    template='plotly_white', height=480,
)
fig.show()
